# Catalog search: 12 test queries

Each query has its own saved output. Uses the existing eval cases, top 10 results and relevance verification (website default). Re-running makes paid API calls. Diagnostic cases are for manual review, not scored accuracy. The original notebook is unchanged.


In [1]:
import json
import importlib
from pathlib import Path
from html import escape

import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, display

from search import pinecone_search

importlib.reload(pinecone_search)
search = pinecone_search.search
from evals.run_eval import assess

ROOT = Path.cwd()
if not (ROOT / "evals/cases.json").exists():
    ROOT = ROOT / "week1_project/catalog_search"
load_dotenv(ROOT / ".env")
CASES = {case["id"]: case for case in json.loads((ROOT / "evals/cases.json").read_text())["cases"]}
TOP_K = 10
VERIFY = True
responses = {}
summary = []


def display_full_text_table(table):
    """Display full values with automatic wrapping and no tooltips."""
    def text(value):
        if isinstance(value, (dict, list, tuple)):
            return json.dumps(value, ensure_ascii=False, default=str)
        return "" if value is None else str(value)

    headers = "".join(f"<th>{escape(str(column))}</th>" for column in table.columns)
    rows = []
    for values in table.itertuples(index=False, name=None):
        cells = []
        for value in values:
            full = escape(text(value), quote=True)
            cells.append(f'<td><div class="full-text-cell">{full}</div></td>')
        rows.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("""<style>
    .full-text-table {border-collapse:collapse; text-align:left;}
    .full-text-table th,.full-text-table td {padding:8px;border:1px solid #ddd;vertical-align:top;}
    .full-text-cell {min-width:120px;max-width:450px;white-space:pre-wrap;overflow-wrap:anywhere;}
    </style>
    <div style="overflow-x:auto"><table class="full-text-table"><thead><tr>"""
    + headers + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table></div>"))


def show_query(case_id, query):
    """Run one search and show its filters, products, review decisions and timings."""
    # Read expectations fresh each time, so edits to cases.json take effect.
    case = next(case for case in json.loads((ROOT / "evals/cases.json").read_text())["cases"] if case["id"] == case_id)
    print("Query:", repr(query))
    print("Expected product IDs:", case.get("expected_ids", []))
    try:
        response = search(query, top_k=TOP_K, verify=VERIFY)
    except Exception as error:
        print(f"Search failed: {type(error).__name__}: {error}")
        summary.append({"case": case_id, "query": query, "status": "error", "results": 0})
        return
    responses[case_id] = response
    outcome = assess(case, response)
    print("Parsed intent:")
    print(json.dumps(response["filters"], indent=2))
    print("Actual mandatory retrieval filter:")
    print(json.dumps(response.get("retrieval_filter"), indent=2))
    print("Clarification:", response.get("clarification"))
    print("Evaluation:", outcome["status"], outcome["failures"])
    products = response["results"]
    print("Results:", len(products))
    if products:
        table = pd.DataFrame(products)
        columns = [c for c in ["product_id", "title", "price", "currency", "category", "score", "retrieval_sources", "match_evidence"] if c in table]
        display_full_text_table(table[columns])
        # Remote thumbnails keep this notebook small; loading depends on the image host.
        cards = "".join(f'<div style="display:inline-block;width:150px;vertical-align:top;margin:8px"><img src="{escape(p.get("image_url") or "", quote=True)}" style="width:130px;height:160px;object-fit:contain"><p>{escape(p.get("title", ""))}</p></div>' for p in products)
        display(HTML(cards))
    if response.get("verification"):
        print("Candidate verification, including rejected products:")
        display_full_text_table(pd.DataFrame(response["verification"]))
    print("Timings:", response.get("timings"))
    print("Retrieval:", response.get("retrieval_info"))
    summary.append({"case": case_id, "query": query, "status": outcome["status"], "results": len(products), "seconds": response.get("timings", {}).get("total_seconds"), "failures": "; ".join(outcome["failures"])})


## 1. unlined shirt

Expected product: Dealia High V Boxy Tee White (10235143159892). Its description says "Good stretch, unlined" and saved lined=no. Shirt is interpreted broadly to include tees. Existing output below is historical; rerun setup and query cells to use the updated case.


In [2]:
query = 'unlined shirt'
show_query('unlined_shirt', query)


Query: 'unlined shirt'
Parsed intent:
{
  "category": "Tops",
  "currency": "USD",
  "price_under": null,
  "attributes": [
    {
      "name": "top_type",
      "values": [
        "button_down_shirt"
      ]
    },
    {
      "name": "lined",
      "values": [
        "no"
      ]
    }
  ],
  "semantic_query": "unlined shirt",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": null
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Tops"
      }
    }
  ]
}
Clarification: None
Evaluation: pass []
Results: 10
Candidate verification, including rejected products:
Timings: {'parse_seconds': 2.909, 'embedding_seconds': 0.786, 'connection_seconds': 0.333, 'vector_search_seconds': 1.174, 'corpus_fetch_seconds': 4.204, 'bm25_fusion_second

,product_id,title,price,currency,category,score,retrieval_sources,match_evidence
0,10192281763924,Quiny Long Sleeve Shirt Black Check,65.0,USD,Tops,0.032787,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Over..."
1,10235143159892,Dealia High V Boxy Tee White,29.0,USD,Tops,0.031498,"[semantic, bm25]","{'source_field': 'description', 'quote': 'High..."
2,10235143487572,Dealia High V Boxy Tee Yellow,29.0,USD,Tops,0.030536,"[semantic, bm25]","{'source_field': 'description', 'quote': 'High..."
3,10235143323732,Dealia High V Boxy Tee Black,29.0,USD,Tops,0.030331,"[semantic, bm25]","{'source_field': 'description', 'quote': 'High..."
4,10172061155412,Star Stallion Oversized Graphic Tee Blue,45.0,USD,Tops,0.029287,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Over..."
5,10249201483860,Nakota V Neck Cinched Tee Butter Yellow,39.0,USD,Tops,0.027652,"[semantic, bm25]","{'source_field': 'description', 'quote': 'V-ne..."
6,10205526720596,Stars Hollow Babydoll Off Shoulder Top Black,45.0,USD,Tops,0.026553,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Baby..."
7,10205526687828,Seasons Change Relaxed Off Shoulder Top Burgundy,49.0,USD,Tops,0.026501,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Off ..."
8,10230866247764,Settle Down Long Sleeve V Neck Top Black,50.0,USD,Tops,0.025992,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Long..."
9,10217651273812,Matelda 3/4 Sleeve Anglaise Cardigan Top Black,49.0,USD,Tops,0.025780,"[semantic, bm25]","{'source_field': 'description', 'quote': '3/4 ..."


,product_id,decision,reason,source_field,quote,title
0,10192281763924,match,The product is a shirt and explicitly describe...,description,Oversized long sleeve buttton-down flannel shi...,Quiny Long Sleeve Shirt Black Check
1,10235143159892,match,"A tee is a type of shirt, and the description ...",description,High v-neck boxy tee with cap sleeves\nGood st...,Dealia High V Boxy Tee White
2,10235143487572,match,"A tee is a type of shirt, and the description ...",description,High v-neck boxy tee with cap sleeves\nGood st...,Dealia High V Boxy Tee Yellow
3,10235143323732,match,"A tee is a type of shirt, and the description ...",description,High v-neck boxy tee with cap sleeves\nGood st...,Dealia High V Boxy Tee Black
4,10230394716244,insufficient_evidence,The item is explicitly a top rather than a shi...,description,"Sleeveless military top\nStand collar, zip-up ...",Ashtonn Sleeveless Military Top Black
5,10249205973076,insufficient_evidence,The item is explicitly a top rather than a shi...,description,Oversized long sleeve raglan top with crew nec...,Settle Down Long Sleeve Raglan Top Blue / White
6,10172061155412,match,"A graphic tee is a type of shirt, and the desc...",description,Oversized graphic tee with crew neckline & cap...,Star Stallion Oversized Graphic Tee Blue
7,10230865264724,insufficient_evidence,The item is explicitly a top rather than a shi...,description,Long sleeve v-neck top with oversized fit\nCre...,Settle Down Long Sleeve V Neck Top Navy
8,10249201483860,match,"A tee is a type of shirt, and the description ...",description,V-neck cinched tee with cap sleeves\nGood stre...,Nakota V Neck Cinched Tee Butter Yellow
9,10249206104148,insufficient_evidence,The item is explicitly a top rather than a shi...,description,Oversized long-sleeve raglan top with crew nec...,Settle Down Long Sleeve Raglan Top Lilac / Dar...


## 2. unlined shirt spaces

Expected product: Dealia High V Boxy Tee White (10235143159892). Its description says "Good stretch, unlined" and saved lined=no. Shirt is interpreted broadly to include tees. Existing output below is historical; rerun setup and query cells to use the updated case.


In [3]:
query = 'unlined  shirt'
show_query('unlined_shirt_spaces', query)


Query: 'unlined  shirt'
Parsed intent:
{
  "category": "Tops",
  "currency": "USD",
  "price_under": null,
  "attributes": [
    {
      "name": "top_type",
      "values": [
        "button_down_shirt"
      ]
    },
    {
      "name": "lined",
      "values": [
        "no"
      ]
    }
  ],
  "semantic_query": "unlined shirt",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": null
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Tops"
      }
    }
  ]
}
Clarification: None
Evaluation: pass []
Results: 10
Candidate verification, including rejected products:
Timings: {'parse_seconds': 1.637, 'embedding_seconds': 0.455, 'connection_seconds': 0.185, 'vector_search_seconds': 0.627, 'corpus_fetch_seconds': 0.001, 'bm25_fusion_secon

,product_id,title,price,currency,category,score,retrieval_sources,match_evidence
0,10192281763924,Quiny Long Sleeve Shirt Black Check,65.0,USD,Tops,0.032787,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Over..."
1,10235143159892,Dealia High V Boxy Tee White,29.0,USD,Tops,0.031498,"[semantic, bm25]","{'source_field': 'description', 'quote': 'High..."
2,10235143487572,Dealia High V Boxy Tee Yellow,29.0,USD,Tops,0.030536,"[semantic, bm25]","{'source_field': 'description', 'quote': 'High..."
3,10235143323732,Dealia High V Boxy Tee Black,29.0,USD,Tops,0.030331,"[semantic, bm25]","{'source_field': 'description', 'quote': 'High..."
4,10172061155412,Star Stallion Oversized Graphic Tee Blue,45.0,USD,Tops,0.029287,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Over..."
5,10249201483860,Nakota V Neck Cinched Tee Butter Yellow,39.0,USD,Tops,0.027652,"[semantic, bm25]","{'source_field': 'description', 'quote': 'V-ne..."
6,10205526720596,Stars Hollow Babydoll Off Shoulder Top Black,45.0,USD,Tops,0.026553,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Baby..."
7,10205526687828,Seasons Change Relaxed Off Shoulder Top Burgundy,49.0,USD,Tops,0.026501,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Off ..."
8,10230866247764,Settle Down Long Sleeve V Neck Top Black,50.0,USD,Tops,0.025992,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Long..."
9,10217651273812,Matelda 3/4 Sleeve Anglaise Cardigan Top Black,49.0,USD,Tops,0.025780,"[semantic, bm25]","{'source_field': 'description', 'quote': '3/4 ..."


,product_id,decision,reason,source_field,quote,title
0,10192281763924,match,The product is a shirt and explicitly states i...,description,Oversized long sleeve buttton-down flannel shi...,Quiny Long Sleeve Shirt Black Check
1,10235143159892,match,"A tee is a type of shirt, and the description ...",description,High v-neck boxy tee with cap sleeves\nGood st...,Dealia High V Boxy Tee White
2,10235143487572,match,"A tee is a type of shirt, and the description ...",description,High v-neck boxy tee with cap sleeves\nGood st...,Dealia High V Boxy Tee Yellow
3,10235143323732,match,"A tee is a type of shirt, and the description ...",description,High v-neck boxy tee with cap sleeves\nGood st...,Dealia High V Boxy Tee Black
4,10230394716244,insufficient_evidence,"The product is an unlined top, but the supplie...",NaN,NaN,Ashtonn Sleeveless Military Top Black
5,10249205973076,insufficient_evidence,"The product is an unlined top, but the supplie...",NaN,NaN,Settle Down Long Sleeve Raglan Top Blue / White
6,10172061155412,match,"A graphic tee is a type of shirt, and the desc...",description,Oversized graphic tee with crew neckline & cap...,Star Stallion Oversized Graphic Tee Blue
7,10230865264724,insufficient_evidence,"The product is an unlined top, but the supplie...",NaN,NaN,Settle Down Long Sleeve V Neck Top Navy
8,10249201483860,match,"A tee is a type of shirt, and the description ...",description,V-neck cinched tee with cap sleeves\nGood stre...,Nakota V Neck Cinched Tee Butter Yellow
9,10249206104148,insufficient_evidence,"The product is an unlined top, but the supplie...",NaN,NaN,Settle Down Long Sleeve Raglan Top Lilac / Dar...


## 3. micro shorts drawstring

Delaynie Navy: micro boxer shorts, elasticated waist, drawstring. Hudsen hem straps do not prove drawstring waist or above-knee length.

In [4]:
query = 'above knee shorts with drawstring waist'
show_query('micro_shorts_drawstring', query)


Query: 'above knee shorts with drawstring waist'
Parsed intent:
{
  "category": "Bottoms",
  "currency": "USD",
  "price_under": null,
  "attributes": [
    {
      "name": "bottom_type",
      "values": [
        "shorts"
      ]
    },
    {
      "name": "pant_length",
      "values": [
        "above_knee"
      ]
    },
    {
      "name": "waistband_style",
      "values": [
        "drawstring"
      ]
    }
  ],
  "semantic_query": "above knee shorts with drawstring waist",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": null
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Bottoms"
      }
    }
  ]
}
Clarification: None
Evaluation: pass []
Results: 2
Candidate verification, including rejected products:
Timings: {'parse_

,product_id,title,price,currency,category,score,retrieval_sources,match_evidence
0,10184947728468,Delaynie Low Rise Micro Boxer Shorts Navy,45.0,USD,Bottoms,0.032258,"[semantic, bm25]","{'source_field': 'description', 'quote': 'elas..."
1,10184947761236,Delaynie Low Rise Micro Boxer Shorts Brown Check,45.0,USD,Bottoms,0.031025,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Elas..."


,product_id,decision,reason,source_field,quote,title
0,10184947728468,match,"The title says “Micro Boxer Shorts,” which con...",description,"elasticated waist, drawstring",Delaynie Low Rise Micro Boxer Shorts Navy
1,10201459884116,insufficient_evidence,The product has a waist but only “adjustable s...,description,Longline cargo shorts with low-rise fit,Hudsen Longline Cargo Shorts Khaki Curve
2,10192281731156,insufficient_evidence,The description states “Elasticated waist” but...,description,Elasticated waist shorts in check print,Quiny Elasticated Waist Shorts Black Check
3,10184947761236,match,"The title says “Micro Boxer Shorts,” which con...",description,"Elasticated waistband, tie drawstring",Delaynie Low Rise Micro Boxer Shorts Brown Check
4,10250817503316,insufficient_evidence,The description states “Elasticated waist” but...,description,Mid-rise wide cargo fleece shorts,Harleah Wide Cargo Fleece Shorts Khaki
5,10224167157844,insufficient_evidence,"The product has an unlined construction, but “...",description,Longline cargo shorts with low-rise baggy fit,Hudsen Longline Cargo Shorts Cream Curve
6,10250817437780,insufficient_evidence,The description states “Elasticated waist” but...,description,Mid-rise wide cargo fleece shorts,Harleah Wide Cargo Fleece Shorts Grey Marle
7,10184947826772,insufficient_evidence,“Mini Shorts” conventionally implies above-kne...,description,Low-rise shorts in check print,Amorah Bubble Mini Shorts Brown Check
8,10224167092308,insufficient_evidence,"The product has an unlined construction, but “...",description,Longline cargo shorts with low-rise baggy fit,Hudsen Longline Cargo Shorts Washed Black Curve
9,10249281699924,insufficient_evidence,“Longline” does not establish above-knee lengt...,description,Longline cargo shorts with low-rise baggy fit,Hudsen Longline Cargo Shorts Turquoise


## 4. drapey shoulder

Ellah one-shoulder drapey dresses are known candidates; wording is ambiguous.

In [5]:
query = 'drape shoulder dress'
show_query('drapey_shoulder', query)


Query: 'drape shoulder dress'
Parsed intent:
{
  "category": "Dresses",
  "currency": "USD",
  "price_under": null,
  "attributes": [],
  "semantic_query": "drape shoulder dress",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": null
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Dresses"
      }
    }
  ]
}
Clarification: None
Evaluation: pass []
Results: 3
Candidate verification, including rejected products:
Timings: {'parse_seconds': 1.384, 'embedding_seconds': 0.915, 'connection_seconds': 0.227, 'vector_search_seconds': 0.811, 'corpus_fetch_seconds': 0.001, 'bm25_fusion_seconds': 0.007, 'verification_seconds': 5.159, 'total_seconds': 8.504}
Retrieval: {'semantic_candidates': 30, 'bm25_candidates': 40, 'fused_candidates': 40,

,product_id,title,price,currency,category,score,retrieval_sources,match_evidence
0,10177465385044,Ellah One Shoulder Drapey Mini Dress Black Curve,59.0,USD,Dresses,0.032787,"[semantic, bm25]","{'source_field': 'title', 'quote': 'Ellah One ..."
1,10184924266580,Ellah One Shoulder Drapey Mini Dress Wine,59.0,USD,Dresses,0.032258,"[semantic, bm25]","{'source_field': 'title', 'quote': 'Ellah One ..."
2,10205315072084,Dreww Long Sleeve Mini Dress Berry,79.0,USD,Dresses,0.026374,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Long..."


,product_id,decision,reason,source_field,quote,title
0,10177465385044,match,The title explicitly describes a one-shoulder ...,title,Ellah One Shoulder Drapey Mini Dress Black Curve,Ellah One Shoulder Drapey Mini Dress Black Curve
1,10184924266580,match,The title explicitly describes a one-shoulder ...,title,Ellah One Shoulder Drapey Mini Dress Wine,Ellah One Shoulder Drapey Mini Dress Wine
2,10147855204436,insufficient_evidence,"The product is explicitly one-shoulder, but th...",description,One shoulder maxi dress in mesh fabric,Spritzia One Shoulder Maxi Dress Plum
3,7836846522452,insufficient_evidence,The description says sleeveless drape maxi dre...,description,Sleeveless drape maxi dress with ruched detail,Ivetta Ruched Maxi Dress Pink
4,7965318348884,insufficient_evidence,The description confirms a one-shoulder mini d...,description,One-shoulder mini dress with side ruching,Darline One Shoulder Mini Dress Yellow
5,9912515166292,reject,The product has an off-shoulder neckline rathe...,description,Maxi dress in ruched & an off-shoulder neckline,Krystyna Off Shoulder Ruched Mesh Maxi Dress B...
6,8420922884180,reject,The dress is sleeveless with a crew neckline a...,description,Pleated sleeveless mini dress,Dreamboat Pleated Mini Dress Yellow
7,10147855237204,reject,The supplied text describes a raglan rib V-nec...,title,Kipp Raglan Rib V Neck Mini Dress Multi,Kipp Raglan Rib V Neck Mini Dress Multi
8,8718498627668,reject,The dress is explicitly strapless and the supp...,description,Strapless bandage mini dress,Alyzza Strapless Bandage Mini Dress Pink
9,10236762882132,reject,The dress is described as plunge with adjustab...,description,Plunge bandage mini dress,Lucinnda Plunge Bandage Mini Dress Purple


## 5. unsupported shoes

Whitespace variant: compare with the corresponding query above.

In [6]:
query = 'wedding shoes'
show_query('unsupported_shoes', query)


Query: 'wedding shoes'
Parsed intent:
{
  "category": null,
  "currency": "USD",
  "price_under": null,
  "attributes": [],
  "semantic_query": "wedding shoes",
  "unsupported_constraints": [],
  "catalog_scope": "unsupported",
  "semantic_fallbacks": [],
  "scope_resolution": "Shoes are not sold; available categories are tops, bottoms, and dresses."
}
Actual mandatory retrieval filter:
null
Clarification: This catalog carries only tops, bottoms, and dresses. The requested product type is not fully supported; please search for a supported item.
Evaluation: pass []
Results: 0
Timings: {'parse_seconds': 1.555, 'total_seconds': 1.555}
Retrieval: None


## 6. unsupported shoes spaces

Whitespace variant: compare with the corresponding query above.

In [7]:
query = ' wedding   shoes '
show_query('unsupported_shoes_spaces', query)


Query: ' wedding   shoes '
Parsed intent:
{
  "category": null,
  "currency": "USD",
  "price_under": null,
  "attributes": [],
  "semantic_query": "wedding shoes",
  "unsupported_constraints": [],
  "catalog_scope": "unsupported",
  "semantic_fallbacks": [],
  "scope_resolution": "Shoes are unsupported; only tops, bottoms, and dresses are sold."
}
Actual mandatory retrieval filter:
null
Clarification: This catalog carries only tops, bottoms, and dresses. The requested product type is not fully supported; please search for a supported item.
Evaluation: pass []
Results: 0
Timings: {'parse_seconds': 1.615, 'total_seconds': 1.615}
Retrieval: None


## 7. comfy pants

No specific positive labeled; detects wrong garment types only. Empty results alone do not establish quality.

In [8]:
query = 'comfy pants'
show_query('comfy_pants', query)


Query: 'comfy pants'
Parsed intent:
{
  "category": "Bottoms",
  "currency": "USD",
  "price_under": null,
  "attributes": [],
  "semantic_query": "comfy pants",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": "Pants are supported bottoms; comfort is a semantic preference, not a hard attribute filter."
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Bottoms"
      }
    }
  ]
}
Clarification: None
Evaluation: pass []
Results: 7
Candidate verification, including rejected products:
Timings: {'parse_seconds': 1.175, 'embedding_seconds': 0.328, 'connection_seconds': 0.185, 'vector_search_seconds': 0.59, 'corpus_fetch_seconds': 0.0, 'bm25_fusion_seconds': 0.007, 'verification_seconds': 5.342, 'total_seconds': 7.628}
Retrieval: {'sema

,product_id,title,price,currency,category,score,retrieval_sources,match_evidence
0,10217650421844,Kimberlyn Jersey Straight Leg Pants Navy,59.0,USD,Bottoms,0.031258,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Mid-..."
1,10217650356308,Kimberlyn Jersey Straight Leg Pants Grey,59.0,USD,Bottoms,0.029877,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Mid-..."
2,10221333086292,Bailor Channel Waistband Capri Pants White,50.0,USD,Bottoms,0.029631,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Wais..."
3,10253328449620,Alba Track Pant Plum,65.0,USD,Bottoms,0.027072,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Low-..."
4,10251424825428,Fleece Baggy Sweatpant Washed Olive,59.0,USD,Bottoms,0.015152,[semantic],"{'source_field': 'description', 'quote': 'Bagg..."
5,10251424858196,Fleece Baggy Sweatpant Washed Pink,59.0,USD,Bottoms,0.014925,[semantic],"{'source_field': 'description', 'quote': 'Bagg..."
6,10251424923732,Baggy Sweatpant Black,59.0,USD,Bottoms,0.014286,[semantic],"{'source_field': 'description', 'quote': 'Bagg..."


,product_id,decision,reason,source_field,quote,title
0,10173549445204,insufficient_evidence,"The item is explicitly pants, but the source d...",title,Kellenna Straight Leg Low Rise Cargo Pants Khaki,Kellenna Straight Leg Low Rise Cargo Pants Khaki
1,10177465352276,insufficient_evidence,"The item is explicitly pants, but the source d...",title,Lunea Concealed Waistband Pants Khaki Curve,Lunea Concealed Waistband Pants Khaki Curve
2,10217650421844,match,The item is pants and the source describes jer...,description,Mid-rise jersey straight-leg pants\nContrast d...,Kimberlyn Jersey Straight Leg Pants Navy
3,10166595584084,insufficient_evidence,"The item is explicitly pants, but the source d...",title,Two Lovers Low Rise Pants Chocolate,Two Lovers Low Rise Pants Chocolate
4,10217650356308,match,The item is pants and the source describes jer...,description,Mid-rise jersey straight-leg pants\nContrast d...,Kimberlyn Jersey Straight Leg Pants Grey
5,10217646391380,insufficient_evidence,"The item is explicitly pants, but the source d...",title,Riyah 3/4 Low Rise Nylon Pants Navy,Riyah 3/4 Low Rise Nylon Pants Navy
6,10221333086292,match,The item is pants and the source describes an ...,description,Waistband capri pants with tie-side detail\nFo...,Bailor Channel Waistband Capri Pants White
7,10181539889236,insufficient_evidence,"The item is explicitly pants, but the source d...",title,Khane Low Rise Wide Leg Pant Black Tall,Khane Low Rise Wide Leg Pant Black Tall
8,10205311729748,insufficient_evidence,"The item is explicitly pants, but the source d...",title,Brea Frill Belted Pants White,Brea Frill Belted Pants White
9,10205312680020,insufficient_evidence,"The item is explicitly pants, but the source d...",title,Brea Frill Belted Pants Iron,Brea Frill Belted Pants Iron


## 8. wide leg jeans

Review expected positives before treating result count as correctness.

In [9]:
query = 'wide leg jeans'
show_query('wide_leg_jeans', query)


Query: 'wide leg jeans'
Parsed intent:
{
  "category": "Bottoms",
  "currency": "USD",
  "price_under": null,
  "attributes": [
    {
      "name": "bottom_type",
      "values": [
        "jeans"
      ]
    },
    {
      "name": "leg_shape",
      "values": [
        "wide"
      ]
    }
  ],
  "semantic_query": "wide leg jeans",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": null
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Bottoms"
      }
    }
  ]
}
Clarification: None
Evaluation: diagnostic []
Results: 5
Candidate verification, including rejected products:
Timings: {'parse_seconds': 1.238, 'embedding_seconds': 0.34, 'connection_seconds': 0.205, 'vector_search_seconds': 0.537, 'corpus_fetch_seconds': 0.0, 'bm25_fusion

,product_id,title,price,currency,category,score,retrieval_sources,match_evidence
0,10236794601556,Daydream Tie Up Jeans Dark Denim,109.0,USD,Bottoms,0.032522,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Low-..."
1,10173499244628,Nikoletta Foldover Jeans Washed Black,85.0,USD,Bottoms,0.031746,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Wide..."
2,10173499048020,Ramos Low Rise Double Button Distressed Wide L...,85.0,USD,Bottoms,0.031319,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Low-..."
3,10181539889236,Khane Low Rise Wide Leg Pant Black Tall,64.0,USD,Bottoms,0.030118,"[semantic, bm25]","{'source_field': 'description', 'quote': 'Pant..."
4,10231188226132,Konstanze High Rise Baggy Jeans Washed Black,85.0,USD,Bottoms,0.027588,"[semantic, bm25]","{'source_field': 'description', 'quote': 'High..."


,product_id,decision,reason,source_field,quote,title
0,10236794601556,match,The description explicitly identifies these as...,description,Low-rise wide-leg tie-up denim jeans,Daydream Tie Up Jeans Dark Denim
1,10173499244628,match,The description explicitly identifies these as...,description,Wide-leg denim jeans with folded-over detail,Nikoletta Foldover Jeans Washed Black
2,10232550948948,reject,These are explicitly described as straight-leg...,description,Mid-rise straight-leg denim jeans,Holidayer Mid Rise Jeans Mid Blue Wash
3,10173499048020,match,The description explicitly identifies these as...,description,Low-rise double-button distressed wide-leg den...,Ramos Low Rise Double Button Distressed Wide L...
4,10181539889236,match,The description explicitly identifies these as...,description,Pants in wide-leg cut & low-rise fit,Khane Low Rise Wide Leg Pant Black Tall
5,10183208173652,reject,These are explicitly described as straight-leg...,description,Denim jeans in straight-leg cut with low-rise fit,Darla Low Rise Straight Leg Jean White Curve
6,10205525016660,reject,These are explicitly described as bootleg deni...,description,Low-rise bootleg denim jeans,Palaces Low Rise Ringspun Bootleg Jeans Dark I...
7,10205525114964,reject,These are explicitly described as straight-leg...,description,Low-rise straight-leg denim jeans,Ellanor Low Rise Ringspun Straight Leg Jean Mi...
8,10177465352276,insufficient_evidence,"The product is explicitly wide-leg pants, but ...",description,Wide-leg pants in a low-rise fit,Lunea Concealed Waistband Pants Khaki Curve
9,10217650421844,reject,These are explicitly described as straight-leg...,description,Mid-rise jersey straight-leg pants,Kimberlyn Jersey Straight Leg Pants Navy


## 9. below knee shorts

Longline alone is insufficient length evidence; no gold positives labeled.

In [10]:
query = 'below knee shorts'
show_query('below_knee_shorts', query)


Query: 'below knee shorts'
Parsed intent:
{
  "category": "Bottoms",
  "currency": "USD",
  "price_under": null,
  "attributes": [
    {
      "name": "bottom_type",
      "values": [
        "shorts"
      ]
    },
    {
      "name": "pant_length",
      "values": [
        "knee"
      ]
    }
  ],
  "semantic_query": "below knee shorts",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": null
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Bottoms"
      }
    }
  ]
}
Clarification: None
Evaluation: diagnostic []
Results: 0
Candidate verification, including rejected products:
Timings: {'parse_seconds': 1.517, 'embedding_seconds': 0.236, 'connection_seconds': 0.197, 'vector_search_seconds': 0.469, 'corpus_fetch_seconds': 0.001, 

,product_id,decision,reason,source_field,quote,title
0,10184947728468,reject,"The product is explicitly described as ""micro ...",title,Delaynie Low Rise Micro Boxer Shorts Navy,Delaynie Low Rise Micro Boxer Shorts Navy
1,10184947761236,reject,"The product is described as ""Boxer shorts"" and...",description,Boxer shorts in check print,Delaynie Low Rise Micro Boxer Shorts Brown Check
2,10201459884116,insufficient_evidence,"""Longline cargo shorts"" suggests a longer styl...",title,Hudsen Longline Cargo Shorts Khaki Curve,Hudsen Longline Cargo Shorts Khaki Curve
3,10217650552916,reject,"""Mini shorts"" imply a short length, contradict...",title,Lydee Lace Up Mini Shorts Brown,Lydee Lace Up Mini Shorts Brown
4,10192281731156,insufficient_evidence,"The item is shorts, but no wording establishes...",description,Elasticated waist shorts in check print,Quiny Elasticated Waist Shorts Black Check
5,10184947826772,reject,"The product is explicitly described as ""Mini S...",title,Amorah Bubble Mini Shorts Brown Check,Amorah Bubble Mini Shorts Brown Check
6,10250817503316,insufficient_evidence,"The product is described as ""wide cargo fleece...",title,Harleah Wide Cargo Fleece Shorts Khaki,Harleah Wide Cargo Fleece Shorts Khaki
7,10224167157844,insufficient_evidence,"""Longline cargo shorts"" suggests a longer styl...",title,Hudsen Longline Cargo Shorts Cream Curve,Hudsen Longline Cargo Shorts Cream Curve
8,10192286318676,reject,"The product is explicitly described as ""micro ...",title,Schneider Foldover Micro Shorts Brown Check,Schneider Foldover Micro Shorts Brown Check
9,10249281699924,insufficient_evidence,"""Longline cargo shorts"" suggests a longer styl...",title,Hudsen Longline Cargo Shorts Turquoise,Hudsen Longline Cargo Shorts Turquoise


## 10. price cap

Whitespace variant: compare with the corresponding query above.

In [11]:
query = 'dresses under $100'
show_query('price_cap', query)


Query: 'dresses under $100'
Parsed intent:
{
  "category": "Dresses",
  "currency": "USD",
  "price_under": 100.0,
  "attributes": [],
  "semantic_query": "dresses under $100",
  "unsupported_constraints": [],
  "catalog_scope": "supported",
  "semantic_fallbacks": [],
  "scope_resolution": null
}
Actual mandatory retrieval filter:
{
  "$and": [
    {
      "embedding_model": {
        "$eq": "text-embedding-3-small"
      }
    },
    {
      "currency": {
        "$eq": "USD"
      }
    },
    {
      "category": {
        "$eq": "Dresses"
      }
    },
    {
      "price": {
        "$lt": 100.0
      }
    }
  ]
}
Clarification: None
Evaluation: fail ['Too few results']
Results: 0
Candidate verification, including rejected products:
Timings: {'parse_seconds': 1.729, 'embedding_seconds': 0.298, 'connection_seconds': 0.187, 'vector_search_seconds': 1.484, 'corpus_fetch_seconds': 0.001, 'bm25_fusion_seconds': 0.007, 'verification_seconds': 4.818, 'total_seconds': 8.523}
Retrieval: {

,product_id,decision,reason,source_field,quote,title
0,9050672103508,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Wildah Linen Smock Maxi Dress Brown,Wildah Linen Smock Maxi Dress Brown
1,7965297639508,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Lucka Babydoll Tie Mini Dress White,Lucka Babydoll Tie Mini Dress White
2,10149115625556,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Eberta Ruched Maxi Dress Butter,Eberta Ruched Maxi Dress Butter
3,7850504781908,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Alanah Mini Dress Blue Ditsy,Alanah Mini Dress Blue Ditsy
4,8420922884180,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Dreamboat Pleated Mini Dress Yellow,Dreamboat Pleated Mini Dress Yellow
5,7840889110612,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Sandat Mini Dress Gingham Grey,Sandat Mini Dress Gingham Grey
6,10166356246612,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Jadira Scarf Maxi Dress Plum,Jadira Scarf Maxi Dress Plum
7,10181716508756,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Renaissance Corset Lace Mini Dress White Black...,Renaissance Corset Lace Mini Dress White Black...
8,7858249695316,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Karlottie Strapless Mini Dress White Burnout,Karlottie Strapless Mini Dress White Burnout
9,9774837366868,insufficient_evidence,"The product is explicitly a dress, but no pric...",title,Odalia Tie Detail Maxi Dress Chocolate,Odalia Tie Detail Maxi Dress Chocolate


## 11. Blue jeans under $100

Expect blue jeans below USD 100; exclude Delaynie Navy shorts and other non-jeans. Uses the VERIFY setting from setup.


In [ ]:
query = 'blue jeans under $100'
show_query('blue_jeans_under_100', query)


## 12. Dresses under $100 — additional check

Same query as the earlier price-cap case, run separately. Expect at least one dress and prices strictly below USD 100. Uses VERIFY from setup; verification has a known missing-price-input bug.


In [ ]:
query = 'dresses under $100'
show_query('dresses_under_100', query)


## Run summary

Statuses check the seed expectations only, not comprehensive relevance. Whitespace variants can be compared using the saved `responses` dictionary.

In [12]:
display_full_text_table(pd.DataFrame(summary))


,case,query,status,results,seconds,failures
0,unlined_shirt,unlined shirt,pass,10,14.866,
1,unlined_shirt_spaces,unlined shirt,pass,10,7.789,
2,micro_shorts_drawstring,above knee shorts with drawstring waist,pass,2,9.119,
3,drapey_shoulder,drape shoulder dress,pass,3,8.504,
4,unsupported_shoes,wedding shoes,pass,0,1.555,
5,unsupported_shoes_spaces,wedding shoes,pass,0,1.615,
6,comfy_pants,comfy pants,pass,7,7.628,
7,wide_leg_jeans,wide leg jeans,diagnostic,5,7.209,
8,below_knee_shorts,below knee shorts,diagnostic,0,6.641,
9,price_cap,dresses under $100,fail,0,8.523,Too few results
